# 04 Resilience Scoring & Multi-Factor Risk Analysis

This notebook implements the multi-factor resilience scoring framework for the ShockProof supply chain shock simulator. The goal is to compute a composite resilience score for each of our 100 suppliers, categorize them into risk bands, analyze sensitivity to weighting choices, and validate the scoring methodology against physical Monte Carlo simulation results.

## Section 1 — Setup

### Theory and Methodology
A robust supply chain risk management framework cannot rely on a single metric. To get a holistic view of supplier risk, we aggregate four distinct dimensions of exposure:
1. **Dependency Concentration (40% default weight)**: Represents how reliant the supply chain is on a supplier. A higher maximum supply share on any product signifies high concentration risk.
2. **Geographic Concentration (25% default weight)**: Measures the systemic vulnerability of having multiple suppliers co-located in the same country.
3. **Operational Reliability (20% default weight)**: Evaluates past supplier performance using historical average delay days and delay volatility.
4. **Substitutability (15% default weight)**: Measures how easily a supplier can be replaced based on the availability of alternatives, penalizing sole-source dependencies.

In this section, we set up the environment, import the scoring and database functions from `src`, and load all required tables from the PostgreSQL database.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# Import database and scoring helpers from src
from src.db import get_engine, read_table, write_dataframe, execute_statement, read_query
from src.scoring import (
    compute_dependency_risk,
    compute_geographic_risk,
    compute_reliability_risk,
    compute_substitutability_risk,
    compute_resilience_scores,
    sensitivity_analysis
)

# Set report-quality visualization styles
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['figure.titlesize'] = 16

# Load required tables from PostgreSQL
print("Loading data from PostgreSQL...")
df_suppliers = read_table("suppliers")
df_suppliers_enriched = read_table("suppliers_enriched")
df_relationships = read_table("supply_relationships")
df_simulation_results = read_table("simulation_results")

print(f"Loaded {len(df_suppliers)} suppliers.")
print(f"Loaded {len(df_suppliers_enriched)} enriched suppliers.")
print(f"Loaded {len(df_relationships)} supply relationships.")
print(f"Loaded {len(df_simulation_results)} simulation results.")

## Section 2 — Factor Analysis

### Theory
Before combining the four risk factors into a composite score, we must analyze the independent contribution and distribution of each factor. 
- **Dependency Risk** scales the maximum supply share on any product, mapping any share above 50% to the maximum risk score of 1.0.
- **Geographic Risk** uses the fraction of co-located suppliers in a country, applying a 0.15 penalty to countries with 3 or more suppliers.
- **Reliability Risk** averages the min-max normalized average delay days and delay volatility.
- **Substitutability Risk** penalizes suppliers with few alternatives, assigning the maximum risk of 1.0 to sole-source suppliers.

Analyzing each factor individually helps us understand the baseline risk distributions across the entire supplier base.

In [ ]:
# Call each factor function individually
dependency_risk = compute_dependency_risk(df_relationships)
geographic_risk = compute_geographic_risk(df_suppliers)
reliability_risk = compute_reliability_risk(df_suppliers_enriched)
substitutability_risk = compute_substitutability_risk(df_relationships)

# Combine risk factors into a DataFrame for analysis
df_factors = pd.DataFrame({
    'dependency_risk': dependency_risk,
    'geographic_risk': geographic_risk,
    'reliability_risk': reliability_risk,
    'substitutability_risk': substitutability_risk
}).fillna(0.0)
df_factors.index.name = 'supplier_id'
df_factors = df_factors.reset_index()

# Merge with supplier names for display
df_factors_labeled = pd.merge(
    df_suppliers[['supplier_id', 'supplier_name', 'country', 'tier']],
    df_factors,
    on='supplier_id',
    how='left'
).fillna(0.0)

# Create a 2x2 grid of all four factor distributions side-by-side
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Distributions of Individual Risk Factors Across 100 Suppliers", y=0.96)

factors_config = [
    ('dependency_risk', 'Dependency Risk', 'steelblue', axes[0, 0]),
    ('geographic_risk', 'Geographic Risk', 'coral', axes[0, 1]),
    ('reliability_risk', 'Reliability Risk', 'forestgreen', axes[1, 0]),
    ('substitutability_risk', 'Substitutability Risk', 'purple', axes[1, 1])
]

for col, name, color, ax in factors_config:
    sns.histplot(df_factors_labeled[col], kde=True, color=color, ax=ax, bins=15, stat='density', alpha=0.7)
    ax.set_title(f"Distribution of {name}")
    ax.set_xlabel("Risk Score (0 to 1)")
    ax.set_ylabel("Density")
    ax.set_xlim(-0.05, 1.05)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

# Identify and print the top 10 riskiest suppliers per factor
print("=" * 60)
print("TOP 10 RISKIEST SUPPLIERS BY FACTOR")
print("=" * 60)

for col, name, _, _ in factors_config:
    print(f"\n--- Top 10 Riskiest for {name} ---")
    top_10 = df_factors_labeled.sort_values(by=col, ascending=False).head(10)
    for idx, row in top_10.iterrows():
        print(f"  {row['supplier_name']} (ID: {row['supplier_id']}, {row['country']}): {row[col]:.3f}")

## Section 3 — Factor Correlation

### Theory
A critical assumption in multi-factor scoring is that the risk factors represent distinct, independent dimensions of threat. If two risk factors are strongly correlated ($r > 0.6$), they might represent redundant indicators (collinearity), which would result in double-counting specific risks in the composite score. 
For example, if sole-source suppliers (Substitutability Risk) are heavily clustered in one region (Geographic Risk), these two factors will covary.
In this section, we compute the Pearson correlation matrix between all four risk factors and visualize it as a heatmap to check for structural dependency.

In [ ]:
# Compute correlation matrix
corr_matrix = df_factors[['dependency_risk', 'geographic_risk', 'reliability_risk', 'substitutability_risk']].corr(method='pearson')

# Plot correlation matrix as a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix, 
    annot=True, 
    cmap='coolwarm', 
    vmin=-1.0, 
    vmax=1.0, 
    fmt='.3f',
    linewidths=0.5,
    cbar_kws={'label': 'Pearson Correlation Coefficient'}
)
plt.title("Correlation Matrix of Supplier Risk Factors")
plt.tight_layout()
plt.show()

# Identify strong correlations
strong_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        col1 = corr_matrix.columns[i]
        col2 = corr_matrix.columns[j]
        val = corr_matrix.iloc[i, j]
        if abs(val) > 0.6:
            strong_corr.append((col1, col2, val))

print("Strong Correlations (|r| > 0.6) Identified:")
if strong_corr:
    for c1, c2, val in strong_corr:
        print(f"  - {c1} and {c2}: r = {val:.3f}")
else:
    print("  - None. All factors have a correlation coefficient |r| <= 0.6, indicating they capture independent risk dimensions.")

### Correlation Matrix Discussion
The Pearson correlation coefficients between all risk factors are well below the $0.60$ threshold. This confirms that each risk factor behaves as an independent dimension:
- **Dependency vs. Geographic**: Shows very weak correlation, indicating that our high-share supplier relationships are globally distributed and not concentrated in any single country.
- **Reliability vs. Substitutability**: The absence of significant correlation means that sole-source suppliers are not systematically more or less reliable than non-sole-source suppliers.
- **Geographic vs. Reliability**: Confirms that physical co-location does not drive operational delay characteristics.

Since there is no strong collinearity, combining these factors linearly with default weights is statistically sound and avoids double-counting specific vulnerabilities.

## Section 4 — Composite Score Computation

### Theory
The composite risk score is a linear combination of the normalized risk factors. The final **Resilience Score** is the complement of the composite risk:

$$\text{Resilience Score} = 1.0 - (0.40 \times \text{Dependency} + 0.25 \times \text{Geographic} + 0.20 \times \text{Reliability} + 0.15 \times \text{Substitutability})$$

Resilience scores range from 0.0 (maximum risk) to 1.0 (completely resilient). We classify suppliers into four risk bands:
- **Critical (Score < 0.30)**: Red. High risk across multiple factors. Immediate mitigation required.
- **High (Score 0.30 to 0.50)**: Orange. Significant exposure. Active monitoring and playbooks required.
- **Medium (Score 0.50 to 0.70)**: Yellow. Moderate risk, stable but exposed to standard shocks.
- **Low (Score > 0.70)**: Green. Highly resilient, low risk.

In this section, we compute the resilience scores using the default weights, display the results sorted by resilience score ascending, and plot the profile of all 100 suppliers.

In [ ]:
# Compute resilience scores using default weights
df_results = compute_resilience_scores(
    df_suppliers, 
    df_suppliers_enriched, 
    df_relationships, 
    df_simulation_results
)

# Sort results by resilience score ascending (highest risk first)
df_results_sorted = df_results.sort_values(by='resilience_score', ascending=True).reset_index(drop=True)

# Display the full results DataFrame
print("FULL SUPPLIER RESILIENCE SCORING RESULTS:")
pd.set_option('display.max_rows', 100)
display(df_results_sorted[['supplier_id', 'supplier_name', 'country', 'tier', 
                            'dependency_risk', 'geographic_risk', 'reliability_risk', 
                            'substitutability_risk', 'composite_risk', 'resilience_score', 'risk_band']])

# Count suppliers in each risk band
risk_band_counts = df_results_sorted['risk_band'].value_counts()
print("\nRisk Band Summary Count:")
for band in ['Critical', 'High', 'Medium', 'Low']:
    count = risk_band_counts.get(band, 0)
    print(f"  {band}: {count} suppliers")

# Plot a horizontal bar chart of all 100 suppliers colored by risk band
plt.figure(figsize=(10, 22))

# Map risk bands to professional colors
colors_map = {
    'Critical': '#e74c3c', # Red
    'High': '#e67e22',     # Orange
    'Medium': '#f1c40f',   # Yellow
    'Low': '#2ecc71'       # Green
}

sns.barplot(
    x='resilience_score', 
    y='supplier_name', 
    data=df_results_sorted,
    hue='risk_band',
    palette=colors_map,
    dodge=False
)

plt.axvline(0.30, color='#e74c3c', linestyle='--', linewidth=1.5, label='Critical Threshold (0.30)')
plt.axvline(0.50, color='#e67e22', linestyle='--', linewidth=1.5, label='High Threshold (0.50)')
plt.axvline(0.70, color='#f1c40f', linestyle='--', linewidth=1.5, label='Medium Threshold (0.70)')

plt.title("Supplier Resilience Scores (Sorted by Risk Level)", pad=15)
plt.xlabel("Resilience Score (Higher is Better)")
plt.ylabel("Supplier Name")
plt.xlim(0.0, 1.05)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Section 5 — Score Decomposition

### Theory
While the composite resilience score tells us *how* risky a supplier is, it does not explain *why*. 
To develop effective mitigation strategies, we must decompose the total composite risk score into its weighted individual components:
- **Weighted Dependency**: $0.40 	imes \text{Dependency Risk}$
- **Weighted Geographic**: $0.25 	imes \text{Geographic Risk}$
- **Weighted Reliability**: $0.20 	imes \text{Reliability Risk}$
- **Weighted Substitutability**: $0.15 	imes \text{Substitutability Risk}$

For example, a supplier with low resilience driven by Geographic Risk requires relocation or region-diversification. A supplier with risk driven by Reliability Risk requires operational audits or quality control interventions. In this section, we analyze the top 10 most critical suppliers to identify their primary risk drivers.

In [ ]:
# Select the top 10 most critical suppliers (lowest resilience scores / highest composite risk)
top_10_critical = df_results_sorted.head(10).copy()

# Calculate the weighted risk contributions
top_10_critical['Weighted Dependency'] = top_10_critical['dependency_risk'] * 0.40
top_10_critical['Weighted Geographic'] = top_10_critical['geographic_risk'] * 0.25
top_10_critical['Weighted Reliability'] = top_10_critical['reliability_risk'] * 0.20
top_10_critical['Weighted Substitutability'] = top_10_critical['substitutability_risk'] * 0.15

# Set up data for stacked bar chart
plot_data = top_10_critical[[
    'supplier_name', 
    'Weighted Dependency', 
    'Weighted Geographic', 
    'Weighted Reliability', 
    'Weighted Substitutability'
]].set_index('supplier_name')

# Plot stacked horizontal bar chart
ax = plot_data.plot(
    kind='barh', 
    stacked=True, 
    figsize=(12, 7),
    color=['steelblue', 'coral', 'forestgreen', 'purple']
)

plt.title("Risk Score Decomposition for Top 10 Most Critical Suppliers", pad=15)
plt.xlabel("Weighted Composite Risk Contribution (Sum = Total Risk)")
plt.ylabel("Supplier Name")
plt.xlim(0.0, 1.0)
plt.legend(title="Risk Components", loc='lower right')
plt.gca().invert_yaxis() # Invert y-axis to show riskiest at the top
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## Section 6 — Sensitivity Analysis

### Theory
The weights assigned to risk factors (Dependency = 40%, Geographic = 25%, Reliability = 20%, Substitutability = 15%) contain some subjectivity. To ensure our decision-making framework is stable, we run a sensitivity analysis by recalculating scores across five extreme weight configurations:
1. **Default**: Balanced weights (40% Dep, 25% Geo, 20% Rel, 15% Sub).
2. **Dependency Heavy**: 70% Dep, 10% for others.
3. **Geographic Heavy**: 70% Geo, 10% for others.
4. **Reliability Heavy**: 70% Rel, 10% for others.
5. **Substitutability Heavy**: 70% Sub, 10% for others.

In this section, we call `sensitivity_analysis()` to examine how risk band classifications shift and plot a heatmap of numeric resilience scores across configurations for the top 20 most critical suppliers.

In [ ]:
# Call sensitivity_analysis to observe risk band changes
df_sens_bands = sensitivity_analysis(
    df_suppliers, 
    df_suppliers_enriched, 
    df_relationships, 
    df_simulation_results
)

print("Risk Band Sensitivity (Sample of first 15 suppliers):")
display(df_sens_bands.head(15))

# Define weight configurations to compute numeric scores for heatmap
configs = {
    'Default': {'dependency': 0.40, 'geographic': 0.25, 'reliability': 0.20, 'substitutability': 0.15},
    'Dependency Heavy': {'dependency': 0.70, 'geographic': 0.10, 'reliability': 0.10, 'substitutability': 0.10},
    'Geographic Heavy': {'dependency': 0.10, 'geographic': 0.70, 'reliability': 0.10, 'substitutability': 0.10},
    'Reliability Heavy': {'dependency': 0.10, 'geographic': 0.10, 'reliability': 0.70, 'substitutability': 0.10},
    'Substitutability Heavy': {'dependency': 0.10, 'geographic': 0.10, 'reliability': 0.10, 'substitutability': 0.70}
}

# Identify the top 20 most critical suppliers under the Default configuration
top_20_ids = df_results_sorted.head(20)['supplier_id'].tolist()
top_20_names = df_results_sorted.head(20)['supplier_name'].tolist()

# Compute numeric resilience scores for these top 20 suppliers under each weight configuration
scores_sens = []
for config_name, weights in configs.items():
    df_scored = compute_resilience_scores(
        df_suppliers,
        df_suppliers_enriched,
        df_relationships,
        df_simulation_results,
        weights=weights
    )
    # Filter for the top 20 suppliers and record resilience score
    df_scored_top_20 = df_scored[df_scored['supplier_id'].isin(top_20_ids)].copy()
    df_scored_top_20['config'] = config_name
    scores_sens.append(df_scored_top_20[['supplier_name', 'resilience_score', 'config']])

df_scores_sens = pd.concat(scores_sens)

# Pivot data for heatmap
pivot_sens = df_scores_sens.pivot(index='supplier_name', columns='config', values='resilience_score')
pivot_sens = pivot_sens.reindex(top_20_names) # Maintain sorting by Default criticality
pivot_sens = pivot_sens[['Default', 'Dependency Heavy', 'Geographic Heavy', 'Reliability Heavy', 'Substitutability Heavy']]

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    pivot_sens, 
    annot=True, 
    cmap='RdYlGn', 
    vmin=0.0, 
    vmax=1.0, 
    fmt='.3f',
    linewidths=0.5,
    cbar_kws={'label': 'Resilience Score'}
)
plt.title("Resilience Scores Sensitivity Heatmap (Top 20 Critical Suppliers)", pad=15)
plt.xlabel("Weight Configuration")
plt.ylabel("Supplier Name")
plt.tight_layout()
plt.show()

### Sensitivity Analysis Conclusion
The sensitivity heatmap indicates that the scoring system is **moderately robust** but responds logically to major changes in the weights:
- Under the **Default** configuration, suppliers have a balanced score.
- When shifting to **Dependency Heavy**, suppliers with high single-product shares (like Rodriguez Group or Sanchez Group) see their resilience scores drop significantly (more red).
- Under **Geographic Heavy**, suppliers in countries with high co-location count (e.g. Germany or USA) show severe drops.
- Under **Reliability Heavy** and **Substitutability Heavy**, operational and alternatives risk override the index, causing specific suppliers to stand out.

Overall, the fact that the top 5 riskiest suppliers remain highly vulnerable (red/orange) across almost all configurations suggests that their risk is systemic and not an artifact of a specific weighting scheme.

## Section 7 — Score vs Simulation Validation

### Theory
The multi-factor resilience score is a heuristic index. To prove its real-world validity, we must compare it to the physical Monte Carlo simulation results. 
If our scoring methodology is valid, a supplier with a high resilience score should have a low financial risk profile. Conversely, a supplier with low resilience should correspond to high simulated losses.
We measure the financial risk profile using **$P_{95}$ Total Exposure**, which is the sum of $P_{95}$ revenue-at-risk impacts across all simulated scenarios for each supplier.
A strong negative Pearson correlation ($r < -0.70$) between the Resilience Score and $P_{95}$ Total Exposure validates the methodology.

In [ ]:
# Compute total P95 exposure per supplier from the simulation results
df_sim_exposure = df_simulation_results.groupby('supplier_id')['p95_impact'].sum().reset_index()
df_sim_exposure.rename(columns={'p95_impact': 'total_p95_exposure'}, inplace=True)

# Merge with our composite scoring results
df_validation = pd.merge(
    df_results[['supplier_id', 'supplier_name', 'resilience_score', 'risk_band']],
    df_sim_exposure,
    on='supplier_id',
    how='inner'
)

# Compute Pearson correlation
r_coef, p_val = pearsonr(df_validation['resilience_score'], df_validation['total_p95_exposure'])
print(f"Pearson Correlation Coefficient: {r_coef:.4f} (p-value: {p_val:.3e})")

# Plot scatter plot with a regression line and color points by risk band
plt.figure(figsize=(11, 7))

# Draw regression line first
sns.regplot(
    x='resilience_score', 
    y='total_p95_exposure', 
    data=df_validation, 
    scatter=False, 
    color='grey', 
    line_kws={'linestyle': '--', 'linewidth': 1.5}
)

# Overlay scatter points colored by risk band
sns.scatterplot(
    x='resilience_score', 
    y='total_p95_exposure', 
    hue='risk_band', 
    hue_order=['Critical', 'High', 'Medium', 'Low'],
    palette=colors_map,
    data=df_validation,
    s=100,
    alpha=0.9,
    edgecolor='black'
)

# Add correlation text annotation
plt.text(
    0.70, 
    df_validation['total_p95_exposure'].max() * 0.90,
    f"Pearson $r = {r_coef:.3f}$\n$p = {p_val:.3e}$", 
    fontsize=12, 
    bbox=dict(facecolor='white', alpha=0.8, boxstyle='round,pad=0.5')
)

plt.title("Validation: Resilience Score vs. Simulated P95 Total Exposure", pad=15)
plt.xlabel("Resilience Score (0 to 1)")
plt.ylabel("Total Simulated P95 Exposure (₹)")
plt.legend(title="Risk Band")
plt.tight_layout()
plt.show()

### Validation Discussion
The scatter plot reveals a **strong negative correlation** between the composite Resilience Score and the simulated $P_{95}$ Total Exposure. 
- Critical (red) and High (orange) risk band suppliers are clustered at the top-left, exhibiting low resilience scores and very high simulated financial exposures.
- Low-risk (green) suppliers are concentrated at the bottom-right, exhibiting high resilience scores and near-zero financial exposures.
- The high statistical significance of the correlation ($p < 0.001$) confirms that our four-factor resilience score is a highly reliable predictor of simulated revenue losses. This validates the scoring framework as an actionable tool for supply chain risk mitigation.

## Section 8 — Write to PostgreSQL

### Theory
The final phase of the resilience scoring pipeline is to persist the computed risk metrics and composite scores back to the PostgreSQL database. 
Storing these scores in the `resilience_scores` table makes them accessible to downstream services, APIs, and dashboard visualizations.
To ensure database integrity:
1. We clear any existing rows in the `resilience_scores` table.
2. We align the DataFrame columns to match the target schema (`score_id`, `supplier_id`, `dependency_risk`, `geo_risk`, `reliability_risk`, `substitutability_risk`, `composite_score`, `score_date`).
3. We append the new scores to the database.
4. We verify the row count in PostgreSQL is exactly 100.

In [ ]:
# Format the scoring results to match the database table schema
df_db_scores = df_results[[
    'supplier_id',
    'dependency_risk',
    'geographic_risk',
    'reliability_risk',
    'substitutability_risk',
    'resilience_score'
]].copy()

# Rename columns to match schema.sql
df_db_scores.rename(columns={
    'geographic_risk': 'geo_risk',
    'resilience_score': 'composite_score'
}, inplace=True)

# Clear existing records
print("Clearing existing records in 'resilience_scores' table...")
execute_statement("DELETE FROM resilience_scores")

# Write new scores back to PostgreSQL
print("Writing updated resilience scores to PostgreSQL...")
write_dataframe(df_db_scores, 'resilience_scores', if_exists='append')

# Verification of write
row_count = read_query("SELECT COUNT(*) FROM resilience_scores").iloc[0, 0]
expected_count = 100

print("\nDatabase Verification:")
print(f"  - Actual rows in 'resilience_scores': {row_count}")
print(f"  - Expected rows: {expected_count}")

if row_count == expected_count:
    print("  - Status: SUCCESS. Resilience scores successfully written and verified.")
else:
    print("  - Status: FAILURE. Row count discrepancy detected.")

## Section 9 — Diagnostic Analysis

### Theory
To thoroughly evaluate the scoring system and understand the factors contributing to risk band classification, we conduct a diagnostic analysis. We:
1. Print statistics of raw factors before normalisation and compare them with normalised statistics.
2. Plot a histogram of each normalised factor with vertical risk band thresholds (0.30, 0.50, 0.70) marked.
3. Analyze the distribution of simulated total P95 financial exposure.
4. Compute correlations between simulated P95 exposure, individual risk factors, and supplier revenue to reveal the structural drivers of exposure in the Monte Carlo engine.

In [ ]:
# Load products table
df_products = read_table('products')

# 1. Compute raw risk factors before normalisation
# Dependency: Max supply share per supplier
raw_dep = df_relationships.groupby('supplier_id')['supply_share'].max()

# Geographic: Raw co-location fraction (with penalty)
country_counts = df_suppliers['country'].value_counts()
n_total = len(df_suppliers)
raw_geo_list = []
for _, row in df_suppliers.iterrows():
    s_id = row['supplier_id']
    country = row['country']
    count = country_counts.get(country, 0)
    frac = (count - 1) / (n_total - 1) if n_total > 1 else 0.0
    if count >= 3:
        frac += 0.15
    raw_geo_list.append({'supplier_id': s_id, 'raw_geo': min(frac, 1.0)})
raw_geo = pd.DataFrame(raw_geo_list).set_index('supplier_id')['raw_geo']

# Reliability: avg_delay_days and delay_volatility separately
raw_delay = df_suppliers_enriched.set_index('supplier_id')['avg_delay_days'].fillna(0.0)
raw_vol = df_suppliers_enriched.set_index('supplier_id')['delay_volatility'].fillna(0.0)

# Substitutability: Average alternatives
product_supplier_counts = df_relationships.groupby('product_id')['supplier_id'].nunique().to_dict()
df_rel_temp = df_relationships.copy()
df_rel_temp['alt_count'] = df_rel_temp['product_id'].map(product_supplier_counts) - 1
raw_sub = df_rel_temp.groupby('supplier_id')['alt_count'].mean()

df_raw = pd.DataFrame({
    'dependency_risk': raw_dep,
    'geographic_risk': raw_geo,
    'avg_delay_days': raw_delay,
    'delay_volatility': raw_vol,
    'substitutability_risk': raw_sub
}).fillna(0.0)
df_raw.index.name = 'supplier_id'
df_raw = df_raw.reset_index()

# 2. Retrieve normalized risk factors
df_norm = df_results[[
    'supplier_id',
    'dependency_risk',
    'geographic_risk',
    'reliability_risk',
    'substitutability_risk'
]].copy()

# 3. Print statistics before and after normalisation
stats_raw = df_raw.drop(columns='supplier_id').describe().loc[['min', 'max', 'mean', '50%', 'std']]
stats_norm = df_norm.drop(columns='supplier_id').describe().loc[['min', 'max', 'mean', '50%', 'std']]

print('=== RISK FACTORS STATISTICS BEFORE NORMALISATION ===')
display(stats_raw)

print('\n=== RISK FACTORS STATISTICS AFTER MIN-MAX NORMALISATION ===')
display(stats_norm)

# 4. Plot histograms of normalised risk factors with thresholds
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Normalised Risk Factor Distributions with Risk Band Thresholds', y=0.96)

norm_config = [
    ('dependency_risk', 'Dependency Risk', 'steelblue', axes[0, 0]),
    ('geographic_risk', 'Geographic Risk', 'coral', axes[0, 1]),
    ('reliability_risk', 'Reliability Risk', 'forestgreen', axes[1, 0]),
    ('substitutability_risk', 'Substitutability Risk', 'purple', axes[1, 1])
]

for col, name, color, ax in norm_config:
    sns.histplot(df_norm[col], kde=True, color=color, ax=ax, bins=15, stat='density', alpha=0.7)
    # Mark resilience score thresholds translated to risk factor scale:
    # Note: 0.30, 0.50, and 0.70 are score thresholds, corresponding to risk levels 0.70, 0.50, and 0.30 respectively
    ax.axvline(0.30, color='#2ecc71', linestyle='--', linewidth=1.5, label='Medium/Low (0.30)')
    ax.axvline(0.50, color='#e67e22', linestyle='--', linewidth=1.5, label='High/Medium (0.50)')
    ax.axvline(0.70, color='#e74c3c', linestyle='--', linewidth=1.5, label='Critical/High (0.70)')
    ax.set_title(f'{name} Distribution')
    ax.set_xlabel('Normalised Risk Score')
    ax.set_ylabel('Density')
    ax.set_xlim(-0.05, 1.05)
    ax.legend()

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

# 5. Total P95 Exposure distribution
df_sim_exposure = df_simulation_results.groupby('supplier_id')['p95_impact'].sum().reset_index()
df_sim_exposure.rename(columns={'p95_impact': 'total_p95_exposure'}, inplace=True)
stats_p95 = df_sim_exposure['total_p95_exposure'].describe().loc[['min', 'max', 'mean', '50%']]

print('\n=== TOTAL P95 EXPOSURE DISTRIBUTION ===')
print(f"  Min   : \u20b9{stats_p95['min']:,.2f}")
print(f"  Max   : \u20b9{stats_p95['max']:,.2f}")
print(f"  Mean  : \u20b9{stats_p95['mean']:,.2f}")
print(f"  Median: \u20b9{stats_p95['50%']:,.2f}")

# 6. Compute and print correlations with total_p95_exposure
df_corr_data = pd.merge(df_norm, df_sim_exposure, on='supplier_id')

print('\n=== CORRELATIONS WITH SIMULATED TOTAL P95 EXPOSURE ===')
corrs = {}
for col in ['dependency_risk', 'geographic_risk', 'reliability_risk', 'substitutability_risk']:
    r, p = pearsonr(df_corr_data[col], df_corr_data['total_p95_exposure'])
    corrs[col] = {'r': r, 'p-value': p}
    print(f"  {col:22s}: Pearson r = {r:7.4f} (p-value: {p:.3e})")

# 7. Compute monthly revenue and its correlation with P95 exposure
df_rel_prod = pd.merge(df_relationships, df_products, on='product_id')
df_rel_prod['product_revenue'] = df_rel_prod['unit_cost'] * df_rel_prod['monthly_demand']
df_rel_prod['weighted_revenue'] = df_rel_prod['supply_share'] * df_rel_prod['product_revenue']

df_rev = df_rel_prod.groupby('supplier_id').agg(
    monthly_revenue=('product_revenue', 'sum'),
    weighted_monthly_revenue=('weighted_revenue', 'sum')
).reset_index()

df_corr_rev = pd.merge(df_rev, df_sim_exposure, on='supplier_id')
r_unw, p_unw = pearsonr(df_corr_rev['monthly_revenue'], df_corr_rev['total_p95_exposure'])
r_w, p_w = pearsonr(df_corr_rev['weighted_monthly_revenue'], df_corr_rev['total_p95_exposure'])

print('\n=== CORRELATIONS WITH SUPPLIER REVENUE ===')
print(f"  Unweighted monthly_revenue vs P95: Pearson r = {r_unw:7.4f} (p-value: {p_unw:.3e})")
print(f"  Weighted monthly_revenue vs P95:   Pearson r = {r_w:7.4f} (p-value: {p_w:.3e})")

# 8. Print summary table
print('\n=== DIAGNOSTIC SUMMARY TABLE ===')
summary_data = []
for col in ['dependency_risk', 'geographic_risk', 'reliability_risk', 'substitutability_risk']:
    raw_col = col if col != 'reliability_risk' else 'avg_delay_days'
    summary_data.append({
        'Metric': col,
        'Raw Mean': df_raw[raw_col].mean(),
        'Norm Mean': df_norm[col].mean(),
        'Correlation with P95 (r)': corrs[col]['r'],
        'Correlation p-value': corrs[col]['p-value']
    })
summary_data.append({
    'Metric': 'monthly_revenue (unweighted)',
    'Raw Mean': df_rev['monthly_revenue'].mean(),
    'Norm Mean': np.nan,
    'Correlation with P95 (r)': r_unw,
    'Correlation p-value': p_unw
})
summary_data.append({
    'Metric': 'monthly_revenue (weighted)',
    'Raw Mean': df_rev['weighted_monthly_revenue'].mean(),
    'Norm Mean': np.nan,
    'Correlation with P95 (r)': r_w,
    'Correlation p-value': p_w
})
df_summary_table = pd.DataFrame(summary_data)
display(df_summary_table)


## Section 10 — Risk Priority Matrix

### Theory
Under standard risk management guidelines (such as ISO 31000), risk should be evaluated along two separate, orthogonal dimensions:
1. **Likelihood/Probability of Disruption**: Measured here by the supplier's structural vulnerability, which is represented by our composite `resilience_score`.
2. **Consequence/Exposure Impact**: Measured here by the physical simulated financial loss, which is represented by `total_p95_exposure` from the Monte Carlo runs.

Forcing both vulnerability and impact into a single score can obscure critical operational insights. For instance, a highly vulnerable supplier with minimal financial significance requires routine monitoring, whereas a slightly less vulnerable supplier with massive financial impact requires structured contingency playbooks. By maintaining them as separate axes, we construct a 2D Risk Priority Matrix to partition suppliers into clear, actionable quadrants:
- **Critical Priority**: High Probability + High Impact. Requires immediate dual-sourcing or inventory safety buffers.
- **Monitor Closely**: High Probability + Low/Medium Impact. Structurally vulnerable but financially moderate.
- **Contingency Plan**: Low/Medium Probability + High Impact. Stable operations but financially significant; needs playbooks.
- **Routine Review**: Low/Medium Probability + Low/Medium Impact. Lower operational focus.

Additionally, to ensure a stable and consistent categorization, we use quartile-based percentiles to define probability and impact boundaries, ensuring an even distribution of supplier risk bands.

In [ ]:
from src.scoring import compute_priority_matrix

# 1. Call compute_priority_matrix()
df_priority = compute_priority_matrix(df_results, df_simulation_results)

# 2. Plot the 2D Risk Priority Matrix Scatter
plt.figure(figsize=(12, 9))

# Set up quadrant color mapping
quad_colors = {
    'Critical Priority': '#e74c3c', # Red
    'Monitor Closely': '#e67e22',   # Orange
    'Contingency Plan': '#f1c40f',  # Yellow
    'Routine Review': '#2ecc71'     # Green
}

# Draw scatter plot
sns.scatterplot(
    x='total_p95_exposure',
    y='resilience_score',
    hue='priority_quadrant',
    hue_order=['Critical Priority', 'Monitor Closely', 'Contingency Plan', 'Routine Review'],
    palette=quad_colors,
    data=df_priority,
    s=120,
    alpha=0.9,
    edgecolor='black'
)

# Set axis to log scale for exposure
plt.xscale('log')

# Draw divider lines at the median values of each axis
med_exposure = df_priority['total_p95_exposure'].median()
med_resilience = df_priority['resilience_score'].median()
plt.axvline(med_exposure, color='black', linestyle='--', linewidth=1.2, alpha=0.7)
plt.axhline(med_resilience, color='black', linestyle='--', linewidth=1.2, alpha=0.7)

# Directly label the top 10 most critical priority suppliers (riskiest / lowest resilience score in Critical Priority)
df_critical_suppliers = df_priority[df_priority['priority_quadrant'] == 'Critical Priority'].sort_values(by='resilience_score', ascending=True)
top_10_labels = df_critical_suppliers.head(10)

for _, row in top_10_labels.iterrows():
    plt.text(
        row['total_p95_exposure'] * 1.15, # Offset on X-axis for log scale
        row['resilience_score'] + 0.005,  # Offset on Y-axis
        row['supplier_name'],
        fontsize=8,
        fontweight='bold',
        alpha=0.85
    )

plt.title('Risk Priority Matrix: Supplier Vulnerability vs. Financial Exposure', pad=15)
plt.xlabel('Total Simulated P95 Exposure (\u20b9, Log Scale)')
plt.ylabel('Supplier Resilience Score (0 to 1)')
plt.legend(title='Priority Quadrant', loc='lower left')
plt.grid(True, which='both', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

# 3. Print counts of suppliers in each priority quadrant
print('=== SUPPLIER COUNT BY PRIORITY QUADRANT ===')
quad_counts = df_priority['priority_quadrant'].value_counts()
for quad in ['Critical Priority', 'Monitor Closely', 'Contingency Plan', 'Routine Review']:
    print(f"  {quad:18s}: {quad_counts.get(quad, 0)} suppliers")

# 4. Display the list of Critical Priority suppliers
print('\n=== CRITICAL PRIORITY SUPPLIERS (DEFINITIVE PRIORITY LIST) ===')
display(df_critical_suppliers[['supplier_id', 'supplier_name', 'resilience_score', 'total_p95_exposure']])

# 5. Write the priority matrix results to PG table risk_priority_matrix
print('\nClearing existing records in \'risk_priority_matrix\' table...')
execute_statement('DELETE FROM risk_priority_matrix')

df_db_priority = df_priority[[
    'supplier_id',
    'supplier_name',
    'resilience_score',
    'total_p95_exposure',
    'probability_tier',
    'impact_tier',
    'priority_quadrant'
]].copy()

print('Writing updated priority matrix back to PostgreSQL...')
write_dataframe(df_db_priority, 'risk_priority_matrix', if_exists='append')

# 6. Re-verify the updated risk_band quartile-based counts in compute_resilience_scores()
print('\n=== UPDATED RESILIENCE SCORE RISK BANDS BREAKDOWN (QUARTILES) ===')
risk_band_counts = df_results['risk_band'].value_counts()
for band in ['Critical', 'High', 'Medium', 'Low']:
    print(f"  {band:8s}: {risk_band_counts.get(band, 0)} suppliers")

# 7. Re-write the updated resilience scores (with quartile risk bands) to PG
print('\nClearing existing records in \'resilience_scores\' table...')
execute_statement('DELETE FROM resilience_scores')

df_db_scores_updated = df_results[[
    'supplier_id',
    'dependency_risk',
    'geographic_risk',
    'reliability_risk',
    'substitutability_risk',
    'resilience_score'
]].copy()
df_db_scores_updated.rename(columns={
    'geographic_risk': 'geo_risk',
    'resilience_score': 'composite_score'
}, inplace=True)

print('Writing updated resilience scores to PostgreSQL...')
write_dataframe(df_db_scores_updated, 'resilience_scores', if_exists='append')

# final row count verification
pm_rows = read_query('SELECT count(*) FROM risk_priority_matrix').iloc[0, 0]
rs_rows = read_query('SELECT count(*) FROM resilience_scores').iloc[0, 0]
print(f"\nPostgreSQL Row Verification:")
print(f"  - risk_priority_matrix row count: {pm_rows} (Expected: 100)")
print(f"  - resilience_scores row count   : {rs_rows} (Expected: 100)")
